# Comparisson of the full range of ATAG Waypoint2050 scenarios and variants

The goal here is to compare all the standard ATAG scenarios, as well as the scenario variants concerning:

- Traffic        : standard scenarios are explored with central air traffic growth, but extra variants are explored with
                low and high growth scenarios

- Scope          : AeroMAPS emissions considers a Well-to-Wake (WTW) scope, while ATAG considers a Tank-to-Wake (TTW)
                scope, a TTW variant is therefore included considering only kerosene combustion emissions (no lifecycle
                approach) and SAF emissions are based on a percentual emissions reductions from the TTW kerosene (p. 57
                fueling net zero report). However, 73.8 gCO2/MJ is considered for kerosene combustion (value from
                https://doi.org/10.1038/s41467-022-35392-1) instead of the 71.82 gCO2/MJ as in the ATAG report (p. 100)

## Load and run custom processes

First, let's create the list of processes that are to be initialized and computed. We'll group scenarios by scenario name (S1, S2, S3), each with an assigned color, where each scenario variant (standard: central and WTW, low WTW, high WTW, central TTW) uses a different linestyle.

In [ ]:
# The scenario ships with the package; copy it somewhere writable before running,
# so this notebook's outputs and regenerated inputs land in ./workdir rather than
# in the installed AeroMAPS.
from aeromaps.utils.scenarios import prepare_scenario
from aeromaps import assemble_processes

%matplotlib widget
from aeromaps import create_process

import gemseo as gm
from aeromaps.utils.functions import custom_logger_config

SCENARIO = prepare_scenario("atag_2nd_edition_full")

custom_logger_config(gm.configure_logger())

variant_suffix_to_fullname = {
    "": "Central (WTW)",
    "-low": "Low growth (WTW)",
    "-high": "High growth (WTW)",
    "-TTW": "Central (TTW)",
}
processes = {}
groups = {}
for i in range(3):
    scenario_idx = i + 1
    groups[f"S{scenario_idx}"] = []
    for suffix, fullname in variant_suffix_to_fullname.items():
        config_file = str(SCENARIO / "config_files" / f"config_s{scenario_idx}{suffix}.yaml")
        process_i = create_process(configuration_file=config_file)

        scenario_fullname = f"S{scenario_idx} - {fullname}"
        processes[scenario_fullname] = process_i
        groups[f"S{scenario_idx}"].append(scenario_fullname)

multi_process = assemble_processes(processes)
multi_process.compute_all()

In [ ]:
multi_process.plot("co2_emissions_comparison", scenario_groups=groups)

In [ ]:
multi_process.plot("cumulative_co2_emissions_comparison", scenario_groups=groups)

In [ ]:
multi_process.plot("energy_consumption_comparison", scenario_groups=groups)

In [ ]:
multi_process.plot("energy_mix_comparison", scenario_groups=groups)

In [ ]:
multi_process.plot("rpk_comparison", scenario_groups=groups)

In [ ]:
multi_process.plot("load_factor_comparison", scenario_groups=groups)

In [ ]:
multi_process.plot("co2_per_rpk_comparison", scenario_groups=groups)

In [ ]:
multi_process.plot("co2_per_rtk_comparison", scenario_groups=groups)

In [ ]:
multi_process.plot("energy_per_ask_comparison", scenario_groups=groups)

In [ ]:
multi_process.plot("energy_per_rtk_comparison", scenario_groups=groups)

In [ ]:
multi_process.plot("drop_in_supply_breakdown", scenario_groups=groups)

In [ ]:
multi_process.plot("hydrogen_supply_comparison", scenario_groups=groups)

In [ ]:
multi_process.plot("electric_supply_comparison", scenario_groups=groups)

In [ ]:
multi_process.plot("biofuel_production_comparison", scenario_groups=groups)

In [ ]:
multi_process.plot("electrofuel_production_comparison", scenario_groups=groups)

In [ ]:
multi_process.plot("biofuel_mix_comparison", scenario_groups=groups)

## Conclusions

Why are AeroMAPS results different to these in the ATAG report? What can be concluded from these differences?

Both consumption and emissions follow qualitatively the trend in ATAG reports. Quantitavely not so much:

- Energy consumption:
    - Kerosene: matches 2025 levels, but AeroMAPS yields to lower initial consumption growth relative to the report (weird since scenario supposes no efficiency gains before 2035). After 2035 consumption peaks, but timing differs between model/report due to differences in the ramp-up of H2/Battery.
    - H2/Electric: mathes similar order of magnitudes, but with different ramp-up dynamics. Relative to the report data, AeroMAPS' sigmoid functions assume a strong uptake in year of Entry-Into-Service (A>R), but a slower initial growth (A<R). On the other hand, this exponential phase is sustained for longer periods in AeroMAPS, while the report transitions to a linear growth (A>R).

- Emissions: CO2 emissions follow a similar pattern to kerosene consumption, however the different lifecycle scopes for kerosene emissions between AeroMAPS (considering full Well-to-Wake) and the Waypoint report (considering only Tank-to-Wake emissions) lead to an almost constant scaling factor between them (close to the ratio between kerosene emission factor with WTW and TTW scopes).


In [ ]:
from aeromaps.utils.functions import clean_notebooks_on_tests

clean_notebooks_on_tests(globals(), force_cleanup=False)